
# 16. Timestamp / Alignment Batch QC Template (Adapted)

Adapted from Natasha sync notebooks (`dot_sync`, `headcams_sync`, `trials_timing`).

Goal: quickly flag sessions where trial/event timing and camera sync look inconsistent.


In [ ]:

import os
from pathlib import Path
import pandas as pd
import numpy as np
import datajoint as dj

if Path.cwd().name == "notebooks":
    os.chdir("..")

repo_root = Path.cwd()
cfg = repo_root / "dj_local_conf.json"
if not cfg.exists():
    raise FileNotFoundError("Missing dj_local_conf.json in repository root")

dj.config.load(str(cfg))
dj.conn()

from adamacs.pipeline import subject, session, scan, trial, event, behavior

INITIALS = os.environ.get("ADAMACS_INITIALS", "NK")
DATE_FROM = os.environ.get("ADAMACS_DATE_FROM", "2025-01-01")


In [ ]:

keys = (
    scan.Scan * session.Session * session.SessionUser * subject.User
    & f'initials = "{INITIALS}"'
    & f'session_datetime >= "{DATE_FROM}"'
).fetch("KEY")

summary_rows = []
for key in keys:
    te = trial.TrialEvent & key
    event_rows = event.Event & key

    trial_count = len(trial.Trial & key)
    trial_event_count = len(te)
    event_count = len(event_rows)
    camsync_count = len(behavior.CamSyncRecording & key)

    event_types = list((event_rows).fetch("event_type")) if event_count else []
    sync_like = sum(1 for e in event_types if "sync" in str(e).lower())

    summary_rows.append(
        {
            **key,
            "trial_count": trial_count,
            "trial_event_count": trial_event_count,
            "event_count": event_count,
            "camsync_count": camsync_count,
            "sync_like_events": sync_like,
        }
    )

summary = pd.DataFrame(summary_rows)
summary


In [ ]:

flags = summary[
    (summary["trial_count"] > 0)
    & (
        (summary["event_count"] == 0)
        | (summary["sync_like_events"] == 0)
        | (summary["camsync_count"] == 0)
    )
]

flags
